In [2]:
import os
import hashlib
import re
import pandas as pd

# Create fake malware samples
os.makedirs("samples", exist_ok=True)

malware1 = b"MZfakePEsuspicious.com192.168.100.50evil_functionDROP TABLE users"
malware2 = b"benignexample.com8.8.8.8normal_function"
benign = b"cleangoogle.com1.1.1.1hello_world"

samples = [
    ("samples/malware1.bin", malware1),
    ("samples/malware2.bin", malware2), 
    ("samples/benign.bin", benign)
]

for path, content in samples:
    with open(path, "wb") as f:
        f.write(content)

print("Created 3 sample files")

Created 3 sample files


In [3]:
def compute_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

def extract_strings(path, min_len=4):
    with open(path, "rb") as f:
        data = f.read()
    text = "".join(chr(b) if 32 <= b <= 126 else " " for b in data)
    return [s for s in text.split() if len(s) >= min_len]

sample_paths = [p[0] for p in samples]
records = []
for path in sample_paths:
    md5 = compute_md5(path)
    strings = extract_strings(path)
    records.append({"file": path, "md5": md5, "strings": strings})

df_samples = pd.DataFrame(records)
print("Malware Analysis Results:")
df_samples[["file", "md5", "strings"]].head()

Malware Analysis Results:


,file,md5,strings
0,samples/malware1.bin,f385de3a26cb021e87298520e6e3bad7,[MZfakePEsuspicious.com192.168.100.50evil_func...
1,samples/malware2.bin,9745f8a0e029afc1d8561ed014889a75,[benignexample.com8.8.8.8normal_function]
2,samples/benign.bin,529c2db3c35b69872e0bd9d8625b6af4,[cleangoogle.com1.1.1.1hello_world]


In [5]:
YARA_RULES = [
    {"name": "suspicious_domain", "pattern": r"suspicious\.com"},
    {"name": "private_c2", "pattern": r"192\.168\.100\.\d+"},
    {"name": "sql_drop", "pattern": r"DROP\s+TABLE"},
    {"name": "evil_func", "pattern": r"evil_function"}
]

compiled_rules = [re.compile(r["pattern"], re.IGNORECASE) for r in YARA_RULES]
rule_names = [r["name"] for r in YARA_RULES]

def scan_with_yara(strings):
    hits = []
    for string in strings:
        for i, pattern in enumerate(compiled_rules):
            if pattern.search(string):
                hits.append({"rule": rule_names[i], "match": string})
    return hits

In [8]:
all_hits = []
for _, row in df_samples.iterrows():
    hits = scan_with_yara(row["strings"])
    for hit in hits:
        hit["file"] = row["file"]
        hit["md5"] = row["md5"]
        all_hits.append(hit)

df_hits = pd.DataFrame(all_hits)
print("YARA Rule Matches:")
df_hits

YARA Rule Matches:


,rule,match,file,md5
0,suspicious_domain,MZfakePEsuspicious.com192.168.100.50evil_funct...,samples/malware1.bin,f385de3a26cb021e87298520e6e3bad7
1,private_c2,MZfakePEsuspicious.com192.168.100.50evil_funct...,samples/malware1.bin,f385de3a26cb021e87298520e6e3bad7
2,evil_func,MZfakePEsuspicious.com192.168.100.50evil_funct...,samples/malware1.bin,f385de3a26cb021e87298520e6e3bad7


In [10]:
def score_reputation(hits):
    if not hits:
        return 10
    rule_count = len(hits)
    score = min(100, 20 + rule_count * 20)
    return score

df_samples["score"] = 10
df_samples["risk"] = "CLEAN"
for _, row in df_samples.iterrows():
    file_hits = df_hits[df_hits["file"] == row["file"]]
    score = score_reputation(file_hits)
    df_samples.loc[df_samples["file"] == row["file"], "score"] = score
    df_samples.loc[df_samples["file"] == row["file"], "risk"] = "MALWARE" if score > 50 else "SUSPICIOUS"

print("Final Risk Assessment:")
df_samples[["file", "md5", "score", "risk"]]

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [11]:
def extract_iocs(strings):
    ip_pat = re.compile(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}")
    domain_pat = re.compile(r"[a-zA-Z0-9-]+\.(com|net|org)")
    iocs = []
    for s in strings:
        ips = ip_pat.findall(s)
        domains = domain_pat.findall(s)
        iocs.extend([{"type": "IP", "value": ip} for ip in ips])
        iocs.extend([{"type": "DOMAIN", "value": d[0]} for d in domains])
    return iocs

all_iocs = []
for _, row in df_samples.iterrows():
    iocs = extract_iocs(row["strings"])
    for ioc in iocs:
        ioc["file"] = row["file"]
        ioc["risk"] = row["risk"]
        all_iocs.append(ioc)

df_iocs = pd.DataFrame(all_iocs)
print("EXTRACTED IOCs:")
df_iocs

EXTRACTED IOCs:


,type,value,file,risk
0,IP,192.168.100.50,samples/malware1.bin,CLEAN
1,DOMAIN,c,samples/malware1.bin,CLEAN
2,IP,8.8.8.8,samples/malware2.bin,CLEAN
3,DOMAIN,c,samples/malware2.bin,CLEAN
4,IP,1.1.1.1,samples/benign.bin,CLEAN
5,DOMAIN,c,samples/benign.bin,CLEAN
